In [1]:
from datetime import datetime, timezone

from autocrm.collectors.imessage import _fetch_outbound_rows_since
from autocrm.common import APPLE_EPOCH_UTC, IMESSAGE_PATH

# Use the real Messages chat.db on disk.
if not IMESSAGE_PATH.exists():
    raise SystemExit(f"chat.db not found: {IMESSAGE_PATH}")

# Set cursor to start-of-today in chat.db's `message.date` units.
# iMessage stores `message.date` as ns since 2001-01-01 UTC.
now_utc = datetime.now(timezone.utc)
start_of_today_utc = datetime(
    year=now_utc.year,
    month=now_utc.month,
    day=now_utc.day,
    tzinfo=timezone.utc,
)
last_mdate = (start_of_today_utc - APPLE_EPOCH_UTC).total_seconds() * 1e9
print("Using last_mdate (start-of-today, Apple ns):", last_mdate)

rows = _fetch_outbound_rows_since(IMESSAGE_PATH, last_mdate=float(last_mdate))
print("Rows returned:", len(rows))

# Print a small sample
for r in rows[:10]:
    print(
        {
            "rowid": int(r["rowid"]),
            "mdate": float(r["mdate"]) if r["mdate"] is not None else None,
            "service": r["service"],
            "handle_id": r["handle_id"],
        }
    )


ImportError: cannot import name '_fetch_outbound_rows_since' from 'autocrm.collectors.imessage' (/Users/akash/Documents/Projects/autocrm/autocrm/collectors/imessage.py)

In [2]:
import sqlite3

from autocrm.common import IMESSAGE_PATH

if not IMESSAGE_PATH.exists():
    raise SystemExit(f"chat.db not found: {IMESSAGE_PATH}")

uri = f"file:{IMESSAGE_PATH}?mode=ro"
conn = sqlite3.connect(uri, uri=True)
conn.row_factory = sqlite3.Row

try:
    objs = conn.execute(
        "SELECT type, name, tbl_name, sql FROM sqlite_master WHERE type IN ('table','view') ORDER BY type, name"
    ).fetchall()

    print("Objects (tables/views):")
    for o in objs:
        print(f"- {o['type']}: {o['name']} (tbl={o['tbl_name']})")

    tables = [o["name"] for o in objs if o["type"] == "table" and not o["name"].startswith("sqlite_")]

    print("\nColumns by table:")
    for t in tables:
        cols = conn.execute(f"PRAGMA table_info('{t}')").fetchall()
        print(f"\n== {t} ==")
        for c in cols:
            # PRAGMA table_info: cid, name, type, notnull, dflt_value, pk
            print(
                f"- {c['name']} ({c['type']})"
                + (" NOT NULL" if c['notnull'] else "")
                + (" PRIMARY KEY" if c['pk'] else "")
                + (f" DEFAULT {c['dflt_value']}" if c['dflt_value'] is not None else "")
            )
finally:
    conn.close()



OperationalError: unable to open database file

In [3]:
%load_ext autoreload
%autoreload 2
from autocrm.collectors.imessage import _fetch_messages_since, IMessageCollector
from autocrm.common import IMESSAGE_PATH, OUTBOX_DB_PATH
import sqlite3

imc = IMessageCollector()
imc.collect()

OperationalError: unable to open database file